# Incident Response Runbook: Pump.fun Insider Key Theft

**Tactic:** Privilege Abuse → Credential Access
**Technique:** T1078 (Valid Accounts) + T1531 (Account Access Removal)
**Severity:** CRITICAL

## Overview

This runbook covers the Pump.fun insider exploit (May 2024) in which a former employee retained
withdraw authority private key material after offboarding. The attacker used a flash loan to
amplify capital, then invoked the privileged withdraw authority to drain bonding curve liquidity
pools, stealing approximately $1.9M before the team paused the protocol.

## MITRE ATT&CK Mapping

| Technique | ID | Description |
|---|---|---|
| Valid Accounts | **T1078** | Former employee used retained valid withdraw-authority keypair |
| Account Access Removal | **T1531** | Failure to revoke privileged program authorities during offboarding |
| Financial Theft | **T1657** | Direct on-chain drain of bonding curve pools via privileged instruction |

## Lateral Movement Analysis

The lateral movement path was enabled entirely by incomplete offboarding:

1. **Employment ends** — employee is offboarded but no process exists to rotate program authorities
2. **Privileged keypair retained** — employee retains the `withdrawAuthority` private key from their local machine
3. **Flash loan amplification** — attacker borrows ~$2M flash loan to artificially inflate bonding curve reserves
4. **Direct protocol admin access** — `withdrawAuthority` instruction requires no multisig, no timelock
5. **Drain** — all bonding curve liquidity withdrawn in a single transaction

**Full lateral movement chain:**
`legitimate employee access → offboarding without key revocation → retained withdraw authority keypair → flash loan exploit → direct protocol drain`

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Identify anomalous withdraw authority transactions
- Attribute the off-hours signing activity
- Quantify drained amounts per bonding curve
- Confirm identity of former employee account


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

print("=" * 60)
print("STEP 1: Detection & Analysis — Pump.fun Insider Exploit")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
splunk_indicators = []
unique_users = set()
source_hosts = set()

# Query Solana RPC logs / on-chain indexer for anomalous withdraw authority calls
print("\n[QUERY] Searching for anomalous withdraw authority transactions...")
splunk_query = '''
index=solana_rpc OR index=onchain_events
(instruction="withdrawAuthority" OR instruction="withdrawFunds" OR program="pump_fun")
| eval hour=strftime("%H", _time)
| where (hour < 6 OR hour > 22)
| stats count, sum(amount_sol) as total_sol by signer, instruction, _time
| sort -total_sol
'''
try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} anomalous withdraw authority events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

for event in splunk_results:
    system_info = {
        'hostname': event.get('signer', 'unknown'),
        'instruction': event.get('instruction', 'unknown'),
        'total_sol': event.get('total_sol', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    unique_users.add(event.get('signer', 'unknown'))
    splunk_indicators.append({
        'type': 'privileged_tx',
        'value': f"signer={event.get('signer')} instruction={event.get('instruction')} amount={event.get('total_sol')} SOL",
        'context': 'Off-hours privileged withdraw authority invocation'
    })

# Query for flash loan activity correlated with drain
print("\n[QUERY] Correlating flash loan activity with protocol drain...")
flash_loan_query = '''
index=onchain_events program="solend" OR program="marginfi" OR program="kamino"
instruction="flashBorrow" OR instruction="flashLoan"
| join tx_hash [search index=onchain_events program="pump_fun" instruction="withdrawFunds"]
| stats count, sum(borrow_amount) as flash_amount by signer, _time
'''
try:
    fl_results = splunk.search_events(flash_loan_query, timeframe="-24h")
    print(f"   Found {len(fl_results)} flash loan + drain correlation events")
    for r in fl_results:
        splunk_indicators.append({
            'type': 'flash_loan_drain',
            'value': f"flash_borrow={r.get('flash_amount')} SOL correlated with drain",
            'context': 'Flash loan used to amplify privileged bonding curve drain'
        })
except Exception as e:
    print(f"   Flash loan correlation query failed: {e}")

# Check MISP for known attacker wallets
print("\n[ENRICHMENT] Checking MISP for insider threat wallet indicators...")
misp_results = []
try:
    for user in unique_users:
        hits = misp.search_iocs(user)
        if hits:
            misp_results.extend(hits)
            print(f"   MISP hit for signer {user[:8]}...: {len(hits)} events")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print("\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Pump.fun Insider Exploit — Withdraw Authority Abuse — {len(splunk_indicators)} events',
        'description': 'Former employee used retained withdraw authority keypair + flash loan to drain bonding curves',
        'severity': 'CRITICAL',
        'tactic': 'Privilege Abuse / Credential Access',
        'technique': 'T1078 Valid Accounts + T1531 Account Access Removal',
        'indicators': splunk_indicators,
        'affected_systems': affected_systems
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n✅ Detection complete:")
print(f"   - Anomalous withdraw authority events: {len(splunk_indicators)}")
print(f"   - Suspected insider wallets: {len(unique_users)}")
print(f"   - MISP threat intel hits: {len(misp_results)}")
print(f"   - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Immediately pause all Pump.fun bonding curve operations
- Revoke withdraw authority from all former employee accounts
- Rotate all program authorities to multisig-controlled keypairs
- Block identified attacker wallet addresses on-chain monitoring


In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
blocked_ips = []
disabled_accounts = []

# 1. Emergency protocol pause
print("\n[CONTAINMENT] Triggering emergency protocol pause...")
try:
    pause_result = shuffle.call_program_instruction(
        program='pump_fun',
        instruction='emergencyPause',
        authority='pause_authority_keypair'
    )
    if pause_result:
        containment_actions.append({'action': 'protocol_pause', 'target': 'pump_fun bonding curves', 'status': 'success', 'timestamp': containment_time})
        print("   ✅ Pump.fun bonding curve operations paused")
except Exception as e:
    print(f"   CRITICAL: Protocol pause failed — manual intervention required: {e}")

# 2. Revoke withdraw authority from former employee accounts
print("\n[CONTAINMENT] Revoking withdraw authority from former employee accounts...")
former_employee_accounts = list(unique_users)
try:
    for account in former_employee_accounts:
        revoke_result = shuffle.revoke_solana_authority(
            program='pump_fun',
            authority_type='withdrawAuthority',
            from_account=account,
            to_account='multisig_address'
        )
        if revoke_result:
            disabled_accounts.append(account)
            containment_actions.append({
                'action': 'authority_revocation',
                'target': account,
                'authority_type': 'withdrawAuthority',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Revoked withdrawAuthority from: {account[:16]}...")
except Exception as e:
    print(f"   Authority revocation failed: {e}")

# 3. Block attacker wallet on-chain monitoring/alerting
print("\n[CONTAINMENT] Adding attacker wallets to blocklist monitoring...")
try:
    for wallet in unique_users:
        alert_result = splunk.add_to_watchlist(wallet, 'insider_threat_wallets')
        if alert_result:
            containment_actions.append({'action': 'wallet_watchlist', 'target': wallet, 'status': 'success', 'timestamp': containment_time})
            print(f"   Added to watchlist: {wallet[:16]}...")
except Exception as e:
    print(f"   Watchlist update failed: {e}")

# 4. Enable enhanced multisig monitoring
print("\n[CONTAINMENT] Enabling enhanced multisig signing monitoring...")
try:
    monitoring_rules = [{
        'name': 'Pump.fun Privileged Instruction Monitoring',
        'query': 'index=onchain_events program="pump_fun" (instruction="withdrawFunds" OR instruction="setAuthority") | alert',
        'alert_threshold': 1,
        'time_window': '1m'
    }]
    splunk.enable_enhanced_monitoring(monitoring_rules)
    print("   Enhanced monitoring enabled for privileged pump.fun instructions")
except Exception as e:
    print(f"   Monitoring setup failed: {e}")

print(f"\n✅ Containment complete:")
print(f"   - Protocol paused: ✓")
print(f"   - Authorities revoked: {len(disabled_accounts)}")
print(f"   - Attacker wallets watchlisted: {len(unique_users)}")


## Phase 3: Eradication

### Objectives
- Deploy program upgrade with new multisig-only authority structure
- Implement timelock on all authority change instructions
- Remove all single-key privileged authorities from the protocol
- Audit entire offboarding checklist for missed key rotations


In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
reset_credentials = []

# 1. Deploy program upgrade with multisig authority structure
print("\n[ERADICATION] Deploying program upgrade with multisig authorities...")
try:
    upgrade_script = '''
    # Deploy upgraded program with:
    # - withdrawAuthority: 3-of-5 multisig (no single signer)
    # - upgradeAuthority: hardware wallet + timelock
    # - pauseAuthority: 2-of-3 multisig for emergency response
    solana program deploy pump_fun_v2.so --keypair multisig_upgrade_authority.json
    '''
    result = shuffle.deploy_program_upgrade('pump_fun', upgrade_script)
    if result:
        eradication_actions.append({'action': 'program_upgrade', 'target': 'pump_fun v2', 'status': 'success', 'timestamp': eradication_time})
        print("   Program upgrade deployed with multisig authorities")
    else:
        print("   Program upgrade failed — manual deployment required")
except Exception as e:
    print(f"   Program upgrade error: {e}")

# 2. Rotate all remaining single-key authorities
print("\n[ERADICATION] Rotating all remaining single-key protocol authorities...")
authority_types = ['withdrawAuthority', 'feeAuthority', 'upgradeAuthority', 'pauseAuthority']
try:
    for auth_type in authority_types:
        rotate_result = shuffle.rotate_program_authority(
            program='pump_fun',
            authority_type=auth_type,
            new_authority_type='multisig'
        )
        if rotate_result:
            reset_credentials.append(auth_type)
            eradication_actions.append({'action': 'authority_rotation', 'target': auth_type, 'status': 'success', 'timestamp': eradication_time})
            print(f"   Rotated {auth_type} to multisig")
except Exception as e:
    print(f"   Authority rotation failed: {e}")

# 3. Audit offboarding checklist — identify missed rotations
print("\n[ERADICATION] Auditing past offboarding events for missed key rotations...")
offboarding_query = '''
index=hr_systems OR index=identity_events
event_type="employee_offboarding" date > "2023-01-01"
| lookup solana_authority_holders by employee_id
| where isnotnull(authority_held)
| stats count by employee_name, authority_type, offboarding_date
'''
try:
    offboarding_results = splunk.search_events(offboarding_query, timeframe="-365d")
    print(f"   Found {len(offboarding_results)} past offboarding events — checking for retained authorities")
    for r in offboarding_results:
        print(f"   ⚠️  Review: {r.get('employee_name','unknown')} had {r.get('authority_type','unknown')} during offboarding on {r.get('offboarding_date','unknown')}")
        eradication_actions.append({'action': 'offboarding_audit_finding', 'target': r.get('employee_name','unknown'), 'status': 'requires_review', 'timestamp': eradication_time})
except Exception as e:
    print(f"   Offboarding audit failed: {e}")

print(f"\n✅ Eradication complete:")
print(f"   - Program upgrade deployed: ✓")
print(f"   - Authorities rotated to multisig: {len(reset_credentials)}")
print(f"   - Offboarding audit completed: ✓")


## Phase 4: Recovery

### Objectives
- Resume protocol operations with upgraded authority structure
- Compensate affected users from team/insurance funds
- Implement on-chain timelock for all future authority changes
- Update incident response procedures


In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_services = []

# 1. Resume bonding curve operations
print("\n[RECOVERY] Resuming bonding curve operations under new authority structure...")
try:
    resume_result = shuffle.call_program_instruction(
        program='pump_fun_v2',
        instruction='resume',
        authority='multisig_pause_authority'
    )
    if resume_result:
        restored_services.append('pump_fun bonding curves')
        recovery_actions.append({'action': 'protocol_resume', 'target': 'pump_fun_v2', 'status': 'success', 'timestamp': recovery_time})
        print("   ✅ Bonding curve operations resumed")
except Exception as e:
    print(f"   Protocol resume failed: {e}")

# 2. Execute user compensation
print("\n[RECOVERY] Processing user compensation for drained bonding curves...")
try:
    compensation_query = '''
    index=onchain_events program="pump_fun" instruction="withdrawFunds"
    | join signer [search index=insider_threat_wallets]
    | stats sum(amount_sol) as drained_sol by user_address
    '''
    drain_results = splunk.search_events(compensation_query, timeframe="-48h")
    total_drained = sum(float(r.get('drained_sol', 0)) for r in drain_results)
    print(f"   Total to compensate: {total_drained:.2f} SOL across {len(drain_results)} affected accounts")
    recovery_actions.append({'action': 'compensation_calculated', 'target': f'{total_drained:.2f} SOL', 'status': 'manual_execution_required', 'timestamp': recovery_time})
except Exception as e:
    print(f"   Compensation calculation failed: {e}")

# 3. Validate new authority structure
print("\n[RECOVERY] Validating new multisig authority structure...")
try:
    validation_result = shuffle.verify_program_authorities('pump_fun_v2', expected_type='multisig')
    if validation_result.get('all_multisig'):
        recovery_actions.append({'action': 'authority_validation', 'target': 'pump_fun_v2', 'status': 'success', 'timestamp': recovery_time})
        print("   ✅ All authorities confirmed as multisig")
    else:
        print(f"   ⚠️  Non-multisig authorities detected: {validation_result.get('single_key_authorities', [])}")
except Exception as e:
    print(f"   Authority validation failed: {e}")

# 4. Restore monitoring
print("\n[RECOVERY] Restoring normal monitoring...")
try:
    splunk.restore_normal_monitoring()
    recovery_actions.append({'action': 'monitoring_restore', 'status': 'success', 'timestamp': recovery_time})
    print("   Monitoring restored")
except Exception as e:
    print(f"   Monitoring restore failed: {e}")

print(f"\n✅ Recovery complete:")
print(f"   - Protocol resumed: ✓")
print(f"   - Services restored: {len(restored_services)}")
print(f"   - Authority structure validated: ✓")


## Phase 5: Post-Incident Activities

### Objectives
- Build formal offboarding key-rotation checklist
- Mandate multisig + timelock for all future protocol authorities
- Share lessons learned across DeFi security community


In [ ]:
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate incident report
print("\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': 'Pump.fun Insider Exploit — IR Report',
        'severity': 'CRITICAL',
        'technique': 'T1078 + T1531',
        'root_cause': 'Incomplete offboarding — withdraw authority private key retained by former employee',
        'timeline': {'detection': detection_time, 'containment': containment_time, 'eradication': eradication_time, 'recovery': recovery_time, 'closure': closure_time},
        'recommendations': [
            'Mandatory key rotation and authority revocation on every employee offboarding',
            'Never use single-key authorities for any privileged protocol instruction',
            'Implement multisig (3-of-5 minimum) for all withdraw/upgrade authorities',
            'Deploy timelock (48–72h minimum) on all authority change instructions',
            'Maintain a live registry of who holds each program authority keypair',
            'Quarterly authority rotation as a preventive practice regardless of personnel changes'
        ]
    }
    report_filename = f"pumpfun_insider_exploit_report_{incident_id}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)
    print(f"   Report written: {report_filename}")
    post_incident_actions.append({'action': 'report_generation', 'target': report_filename, 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   Report generation failed: {e}")

# 2. Build offboarding runbook
print("\n[POST-INCIDENT] Creating formal offboarding key-rotation checklist...")
offboarding_checklist = {
    'steps': [
        'Identify all program authorities held by departing employee (withdrawAuthority, feeAuthority, upgradeAuthority)',
        'Rotate each authority to a new multisig controlled keypair before last day',
        'Revoke all SSH, GitHub, and cloud provider access',
        'Rotate any shared secrets the employee had access to',
        'Verify on-chain that old public key has no remaining authority over any program',
        'Document rotation in security audit log'
    ]
}
checklist_filename = "offboarding_key_rotation_checklist.json"
with open(checklist_filename, 'w') as f:
    json.dump(offboarding_checklist, f, indent=2)
print(f"   Offboarding checklist written: {checklist_filename}")
post_incident_actions.append({'action': 'checklist_creation', 'target': checklist_filename, 'status': 'success', 'timestamp': closure_time})

# 3. Close IRIS case
print("\n[POST-INCIDENT] Closing incident case...")
try:
    iris.close_case(incident_id, {'status': 'closed', 'resolution': 'Program upgraded to multisig, users compensated'})
    print(f"   IRIS case closed: {incident_id}")
except Exception as e:
    print(f"   Case closure failed: {e}")

print(f"\n✅ Post-incident activities complete:")
print(f"   - Report generated ✓")
print(f"   - Offboarding checklist created ✓")
print(f"   - Case closed: {incident_id}")
print(f"\n🔒 Pump.fun Insider Exploit IR Complete")


## Summary

The Pump.fun insider exploit demonstrates the critical importance of treating privileged keypair
access as a time-limited credential requiring revocation upon any personnel change.

### Key Takeaways
- Single-key program authorities are an unacceptable risk for production DeFi protocols
- Offboarding must include explicit cryptographic key rotation, not just account deactivation
- Flash loans can amplify insider exploits — protocol pausing mechanisms must be faster than block time
- Multisig with timelock eliminates single-point-of-failure for privileged instructions

### Preventive Architecture
- All protocol authorities: 3-of-5 multisig minimum
- Authority changes: 72h timelock
- Offboarding checklist: mandatory key rotation sign-off before final paycheck


## References

- https://rekt.news/pumpfun-rekt/ — Rekt.news incident analysis
- https://attack.mitre.org/techniques/T1078/ — MITRE T1078: Valid Accounts
- https://attack.mitre.org/techniques/T1531/ — MITRE T1531: Account Access Removal
- https://docs.squads.so/ — Squads multisig for Solana program authorities
- https://www.sec.gov/litigation/litreleases/2024/lr25961.htm — SEC insider threat framework
- https://github.com/solana-labs/solana-program-library/tree/master/timelock — Solana timelock patterns
